# 0. Setting Up The Env

In [1]:

!uv pip install bs4 
# langchainhub langchain_community tiktoken langchain-openai langchainhub chromadb langchain

Using Python 3.13.3 environment at: D:\01 Work\10-New-Learnings\.venv
Audited 1 package in 93ms


## 0.1 Import

In [2]:
import os
from dotenv import load_dotenv

In [3]:
import bs4

# LangChain core
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# Hub
from langsmith import Client
# Loaders & Vector DBs 
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma

# OpenAI 
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
from typing import Literal
# from langchain_core.pydantic_v1 import BaseModel, Field

# 1. Logical & Semantic Routing

* Decompose question to the right data source
* One way, **Logical Routing**, where we give LLMs the knowledge of the LLMs in our didposal and let LLM ro reason which one to apply
* Another way, **Semantic Routing**, where we use prompts, which helps it decide
* There their is a user prompt, and each prompt for each topic, and based on the similarity, topics gets routed


Data model

In [ ]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

# Data model
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )

# LLM with function call 
llm = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

# Prompt 
system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# Define router 
router = prompt | structured_llm

NameError: name 'BaseModel' is not defined

In [ ]:
question = """Why doesn't the following code work:

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})

# 2. Query Construction

* The fastest Query structing is converst this NLP into structured query that can be applied on the metadat filters on the vectore store
* These are applied on the chunks that are indexed
* This is very helpful because it starts from 
    i) Unstructured Input 
    ii) Structured Query Output
    iii) Arbitary Schema that on can use